In [2]:
%pip install "numpy<2.0.0" --force-reinstall

Note: you may need to restart the kernel to use updated packages.Collecting numpy<2.0.0
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   --- ------------------------------------ 1.3/15.5 MB 13.4 MB/s eta 0:00:02
   ----------- ---------------------------- 4.5/15.5 MB 14.9 MB/s eta 0:00:01
   ---------------------- ----------------- 8.7/15.5 MB 16.3 MB/s eta 0:00:01
   ----------------------------- ---------- 11.5/15.5 MB 16.0 MB/s eta 0:00:01
   ----------------------------------- ---- 13.6/15.5 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------  15.5/15.5 MB 14.1 MB/s eta 0:00:01
   ---------------------------------------- 15.5/15.5 MB 13.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6



  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [1]:
import pandas as pd
import glob
import os

# 1. Localizar los archivos dentro de Rail y Sleeper
archivos_rail = glob.glob("Rail/*") + glob.glob("Rail/**/*", recursive=True) + glob.glob("rail/*")
archivos_sleeper = glob.glob("Sleeper/*") + glob.glob("Sleeper/**/*", recursive=True) + glob.glob("sleeper/*")

exts = ('.csv', '.txt')
archivos_rail = [f for f in archivos_rail if os.path.isfile(f) and f.endswith(exts)]
archivos_sleeper = [f for f in archivos_sleeper if os.path.isfile(f) and f.endswith(exts)]

print(f"Archivos encontrados en Rail: {len(archivos_rail)}")
print(f"Archivos encontrados en Sleeper: {len(archivos_sleeper)}")

lista_dfs = []

def procesar_csv_sin_duplicados(ruta, posicion):
    try:
        df = pd.read_csv(ruta, dtype=str, on_bad_lines='skip')
        df = df.dropna(how='all', axis=1)
        
        nuevas_columnas = []
        vistos = set()
        
        for col in df.columns:
            col_lower = str(col).lower()
            
            if ('date' in col_lower or 'time' in col_lower) and 'timestamp' not in vistos:
                nombre = 'timestamp'
            elif ('axis_x' in col_lower or 'x_reading' in col_lower) and 'X_reading_deg' not in vistos:
                nombre = 'X_reading_deg'
            elif ('axis_y' in col_lower or 'y_reading' in col_lower) and 'Y_reading_deg' not in vistos:
                nombre = 'Y_reading_deg'
            elif ('axis_z' in col_lower or 'z_reading' in col_lower) and 'Z_reading_deg' not in vistos:
                nombre = 'Z_reading_deg'
            elif 'temp' in col_lower and 'temperature_C' not in vistos:
                nombre = 'temperature_C'
            else:
                nombre = col
                
            vistos.add(nombre)
            nuevas_columnas.append(nombre)

        df.columns = nuevas_columnas
        df = df.loc[:, ~df.columns.duplicated()]
        
        if 'sensor_id' not in df.columns:
            df['sensor_id'] = posicion
        if 'row_status' not in df.columns:
            df['row_status'] = 'OK'
        if 'incident_id' not in df.columns:
            df['incident_id'] = 0

        return df
    except Exception as e:
        print(f"Error procesando {ruta}: {e}")
        return None

for f in archivos_rail:
    res = procesar_csv_sin_duplicados(f, 'Rail')
    if res is not None:
        lista_dfs.append(res)

for f in archivos_sleeper:
    res = procesar_csv_sin_duplicados(f, 'Sleeper')
    if res is not None:
        lista_dfs.append(res)

# 2. CONCATENACIÓN, LIMPIEZA Y EXPORTACIÓN DEL CSV FINAL
if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)
    
    # Conversión de columnas de sensores a valores numéricos
    cols_numericas = ['X_reading_deg', 'Y_reading_deg', 'Z_reading_deg', 'temperature_C']
    for col in cols_numericas:
        if col in df_final.columns:
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

    # Convertir timestamp a fecha para ordenar correctamente
    if 'timestamp' in df_final.columns:
        df_final['timestamp'] = pd.to_datetime(df_final['timestamp'], errors='coerce')
        df_final = df_final.sort_values('timestamp')

    # Imputación de nulos (rellenar vacíos con el valor anterior/posterior)
    df_final = df_final.ffill().bfill()
    
    # Si quedase algún nulo residual, rellenar con 0
    df_final = df_final.fillna(0)
    
    # Crear carpeta data/ si no existe
    os.makedirs("data", exist_ok=True)
    ruta_salida = "data/tiltmonitor_incidence_series_24h.csv"
    
    # Guardar el CSV consolidado y limpio
    df_final.to_csv(ruta_salida, index=False)
    
    print("\n¡ÉXITO COMPLETO!")
    print(f"Archivo compilado y limpio en: {os.path.abspath(ruta_salida)}")
    print(f"Filas unificadas: {df_final.shape[0]}, Columnas: {df_final.shape[1]}")
    print(f"Valores nulos (NaN) restantes: {df_final.isna().sum().sum()}")
else:
    print("\nNo se pudieron leer los archivos de las carpetas Rail/Sleeper.")

Archivos encontrados en Rail: 78
Archivos encontrados en Sleeper: 78

¡ÉXITO COMPLETO!
Archivo compilado y limpio en: c:\Users\maryp\Desktop\Carrera\practicas\prueba_world\data\tiltmonitor_incidence_series_24h.csv
Filas unificadas: 123066, Columnas: 294
Valores nulos (NaN) restantes: 0
